# Автоматизация генерации признаков: итоговая проверка
Последовательный запуск на сохранённых таблицах 5000 пользователей.
Исторические результаты находятся в `reports/historical/` и не перезаписываются.
Здесь повторно обучаются два классификатора с параметрами по умолчанию и два с сохранёнными параметрами Optuna.
Повторная оценка параметров не восстанавливает историю исходного поиска Optuna.
Полная исходная исследовательская история сохранена в `T-Lab.original.ipynb`.


In [ ]:
from pathlib import Path
import os, sys, json, platform, importlib.metadata
import numpy as np
import pandas as pd
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir():
    raise RuntimeError("Запустите ноутбук из корня проекта или notebooks/")
sys.path.insert(0, str(ROOT))
from src.models import fit_eval, feature_columns
from src.metrics import rank_metrics, paired_bootstrap
DATA = Path(os.environ.get("RECSYS_DATA_DIR", ROOT / "data"))
OUT = ROOT / "reports" / "recheck"
OUT.mkdir(parents=True, exist_ok=True)
RUN_SEARCH = False  # True запускает новый поиск, 30 trials для каждого набора
N_TRIALS = 30
N_BOOT = 2000
SEED = 42


In [ ]:
tables = {}
for split, stem in (("train", "Xtr"), ("val", "Xva"), ("test", "Xte")):
    A = pd.read_parquet(DATA / f"{stem}_5k.parquet").reset_index(drop=True)
    B = pd.read_parquet(DATA / f"{stem}_a_5k.parquet").reset_index(drop=True)
    assert A[["uid", "item_id", "label"]].equals(B[["uid", "item_id", "label"]]), split
    assert A.label.isin([0, 1]).all()
    assert A.ui_likes_all.eq(0).all() and B.ui_likes_all.eq(0).all()
    tables[split] = (A, B)
Xtr, Xtr_a = tables["train"]
Xva, Xva_a = tables["val"]
Xte, Xte_a = tables["test"]
manual = feature_columns(Xtr, ("ui_likes_",))
auto = feature_columns(Xtr_a, ("ui_likes_", "ui_cr_"))
assert (len(manual), len(auto)) == (61, 151)
summary = []
for split, (A, B) in tables.items():
    summary.append(dict(split=split, rows=len(A), users=A.uid.nunique(),
                        positive_users=int(A.groupby("uid").label.sum().gt(0).sum())))
print(pd.DataFrame(summary).to_string(index=False))
pd.DataFrame(summary).to_csv(OUT / "data_audit.csv", index=False)


## Optuna
Сохранённый JSON подтверждает наличие результатов настройки, но не содержит числа trials,
истории, seed и пространства поиска. Равенство исторических бюджетов не доказано.
Новый протокол использует один бюджет и пространство для обоих наборов, NDCG@10 на val,
фиксированный seed и последовательный поиск. Test не передаётся в `tune`.
Early stopping классификатора использует binary logloss на val; цель Optuna — NDCG@10.
Для нового запуска выберите уникальный STUDY_TAG; существующая study намеренно не перезаписывается.


In [ ]:
historical = json.loads((ROOT / "reports/historical/optuna_results.json").read_text())
params = historical["best_params"]
if RUN_SEARCH:
    from src.tuning import tune
    STUDY_TAG = "new-30trials-seed42"
    params = {
        "manual+ar/al": tune(Xtr, Xva, manual, f"{STUDY_TAG}-manual", OUT, N_TRIALS, SEED),
        "auto-all": tune(Xtr_a, Xva_a, auto, f"{STUDY_TAG}-auto", OUT, N_TRIALS, SEED),
    }
(OUT / "params_used.json").write_text(json.dumps(params, indent=2))


In [ ]:
models, rows, test_scores = {}, [], {}
for feature_set, feats, tr, va, te in (
    ("manual+ar/al", manual, Xtr, Xva, Xte),
    ("auto-all", auto, Xtr_a, Xva_a, Xte_a),
):
    for tuned in (False, True):
        name = ("TUNED " if tuned else "default ") + feature_set
        model, result = fit_eval(name, feats, tr, va, te, params[feature_set] if tuned else None)
        models[name] = model
        model.booster_.save_model(str(OUT / f"{name.replace(' ', '_').replace('/', '_')}.txt"))
        for split, metric in result.items():
            rows.append(dict(model=name, split=split, ndcg10=metric["ndcg@10"], recall10=metric["recall@10"]))
        test_scores[name] = model.predict_proba(te[feats])[:, 1]
        pd.DataFrame({"feature":feats, "gain":model.booster_.feature_importance("gain")}).sort_values(
            "gain", ascending=False).to_csv(OUT / f"importance_{feature_set.replace('/', '_')}_{tuned}.csv", index=False)
results = pd.DataFrame(rows)
results.to_csv(OUT / "results_recheck.csv", index=False)
print(results.to_string(index=False))


In [ ]:
ref = "TUNED manual+ar/al"
draws = paired_bootstrap(Xte, test_scores, ref, n=N_BOOT, seed=SEED)
base = rank_metrics(Xte, test_scores[ref])["ndcg@10"]
diffs = {}
for name, values in draws.items():
    low, high = np.percentile(values, [2.5, 97.5])
    diffs[name] = dict(point_delta=rank_metrics(Xte, test_scores[name])["ndcg@10"]-base,
                       bootstrap_mean=float(np.mean(values)), ci95=[float(low), float(high)])
payload = dict(ref=ref, metric="NDCG@10, test", n_boot=N_BOOT, seed=SEED,
               population="users with positive test labels", diffs=diffs)
(OUT / "bootstrap_recheck.json").write_text(json.dumps(payload, indent=2))
versions = {m: importlib.metadata.version(m) for m in
            ("numpy", "pandas", "lightgbm", "optuna", "scikit-learn", "pyarrow")}
(OUT / "run_metadata.json").write_text(json.dumps(dict(python=platform.python_version(),
    versions=versions, run_search=RUN_SEARCH, data_summary=summary, seed=SEED,
    tie_policy="stable input order for exact ties", n_boot=N_BOOT), indent=2))
print(json.dumps(payload, indent=2))


## Границы проверки
Таблицы не содержат timestamp кандидатов и cutoff; из кешей нельзя доказать отсутствие временной утечки.
При пересборке данных нужно исключить последний час train перед val и последний час наблюдений test:
метка использует будущий час, а исходный код этого не учитывает. Старые метрики относятся к прежнему протоколу.
Строка — прослушивание, повторные uid/item допустимы; метрика оценивает события, а не уникальные треки.
Отсутствие значимой разницы не доказывает эквивалентность или «не хуже».
Ранкер и отбор top-37 сохранены как исторические эксперименты, в этом проверочном запуске не переобучаются.
